In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:130px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

<b><font size=5 color=red>ch15 데이터베이스 연동</font></b>

# 1절. SQLite 데이터 베이스 연동
- 파이썬 자체에 포함.
- C언어 기반의 라이브러리로 속도가 빠른 장점이 있으나 이름에서 알 수 있듯이 lite(light의 비표준 표기 중 하나) 버젼으로 보안성 등은 문제 발생 가능 => 프로토타입용
- SQLite 데이터 베이스는 별도의 DBMS 없이 SQL을 이용하여 DB 엑세스 가능한 디스크 기반 DB
- SQLite는 프로토타입(시제품) 용도
- 프로젝트 진행 단계 : 요구사항(도메인) 분석 - 설계(DB, 기능 등) - 구현 - 테스트 - 고객 인계,배포 - 유지보수 => 설계 전 간소화 버젼으로 프로토타입 활용
- 설계 전 : SQLite <-> 설계 부터 DBMS : Oracle, MySql, MariaDB, mongoDB(데이터의 인스턴스 개수가 달라도 저장이 된다) ....
- C 라이브러리
- <a href ='https://sqlitebrowser.org/dl/' color = 'blue'>db browser for sqlite</a>
## 1.1 SQLite 패키지

In [6]:
import sqlite3
sqlite3.sqlite_version # pandas의 경우 pd.__version__

'3.40.1'

## 1.2 데이터 베이스 연결

In [7]:
# DB 연결
conn = sqlite3.connect('data/ch15_example.db') # update, delete, insert 시 무조건 close() => 하지 않으면 향후 에러 발생
conn # 0 이면 에러, 객체형 반환 시 생성 성공

In [8]:
# 커서 객체(Sql문 실행 -> 결과 조회(list형태, 정수...)) 생성
cursor = conn.cursor()
cursor # 0 이면 에러, 객체형 반환 시 생성 성공

### INSERT, UPDATE, DELETE 전송

In [10]:
# 커서 객체를 통해 내용 전송
cursor.execute('''
    CREATE TABLE MEMBER(
        NAME TEXT,
        AGE INT,
        EMAIL TEXT
    )
''')

In [11]:
cursor.execute('''
    INSERT INTO MEMBER VALUES('홍길동',20,'h@h.com')
''')
print('수행결과 행수 :', cursor.rowcount)

수행결과 행수 : 1


In [31]:
sql = "INSERT INTO MEMBER VALUES('박길동',30,'k@k.com')"
print(sql)
cursor.execute(sql)
print('전송 결과 :', cursor.rowcount)

INSERT INTO MEMBER VALUES('박길동',30,'k@k.com')
전송 결과 : 1


In [32]:
# 커밋
conn.commit() # (반) conn.rollback() DML에서만 가능

### SELECT 전송

In [38]:
cursor.execute('''
    SELECT NAME, AGE+1 NEXTAGE, EMAIL FROM MEMBER
''')

In [39]:
# SELECT 전송
#  select문 실행 결과를 받는 함수
#  cursor.fetchone() : 결과를 한 행만 받을 때 (튜플형태 : ('홍길동', 21, 'h@h.com') ) fetch는 => 한 번 작동하면 다음 데이터로 커서가 이동하여 작동하므로 주의
                                                                                            # 결과만 보면 리스트의 pop과 동일
#  cursor.fetchall() : 결과를 모두 받을 때 [리스트 ('홍길동', 21, 'h@h.com'), ('김길동', 31, 'k@k.com')]
#  cursor.fetchmany(n) : 결과를 n행 받을 때 [리스트 ('홍길동', 21, 'h@h.com'), ('김길동', 31, 'k@k.com')]
# cursor.description : header 내용ㅇㄹ 포함한 내용 list
# INSERT, UPDATE, DELETE 전송 : cursor.rowcount
print(cursor.fetchall())

[('홍길동', 21, 'h@h.com'), ('김길동', 31, 'k@k.com'), ('마길동', 31, 'k@k.com'), ('윤길동', 31, 'k@k.com'), ('박길동', 31, 'k@k.com')]


In [40]:
print(cursor.fetchall())

[]


In [41]:
cursor.execute('''
    SELECT NAME, AGE+1 NEXTAGE, EMAIL FROM MEMBER ORDER BY NEXTAGE
''')
members = cursor.fetchmany(3)
members

[('홍길동', 21, 'h@h.com'), ('김길동', 31, 'k@k.com'), ('마길동', 31, 'k@k.com')]

In [51]:
cursor.execute('''
    SELECT NAME, AGE+1 NEXTAGE, EMAIL FROM MEMBER ORDER BY NEXTAGE
''')
members = cursor.fetchone()
members

('홍길동', 21, 'h@h.com')

In [52]:
cursor.fetchall()

[('김길동', 31, 'k@k.com'),
 ('마길동', 31, 'k@k.com'),
 ('윤길동', 31, 'k@k.com'),
 ('박길동', 31, 'k@k.com')]

In [ ]:
# 데이터 한 줄씩 가져오기

In [54]:
cursor.execute('''
    SELECT NAME, AGE+1 NEXTAGE, EMAIL FROM MEMBER ORDER BY NEXTAGE
''')
members = []
while True :
    member = cursor.fetchone()
    if member is None :
        break
    print(member)
    members.append({'name' : member[0], 'age' : member[1], 'email' : member[2]})
import pandas as pd
pd.DataFrame(members)

('홍길동', 21, 'h@h.com')
('김길동', 31, 'k@k.com')
('마길동', 31, 'k@k.com')
('윤길동', 31, 'k@k.com')
('박길동', 31, 'k@k.com')


,name,age,email
0,홍길동,21,h@h.com
1,김길동,31,k@k.com
2,마길동,31,k@k.com
3,윤길동,31,k@k.com
4,박길동,31,k@k.com


In [63]:
cursor.execute('''
    SELECT NAME, AGE+1 NEXTAGE, EMAIL FROM MEMBER ORDER BY NEXTAGE
''')
members = cursor.fetchall()
df = pd.DataFrame(members, columns = [i[0] for i in cursor.description])
df

,NAME,NEXTAGE,EMAIL
0,홍길동,21,h@h.com
1,김길동,31,k@k.com
2,마길동,31,k@k.com
3,윤길동,31,k@k.com
4,박길동,31,k@k.com


In [61]:
# select문을 수행한 field 정보
cursor.description
[i[0] for i in cursor.description]

['NAME', 'NEXTAGE', 'EMAIL']

## 1.3 SQL 구문에 파라미터 사용하기
- qmark 방식(DB에 따라 불가한 경우 있음. 대표적으로 ORACLE)
- named 방식(추천)

In [64]:
conn = sqlite3.connect('data/ch15_example.db')
cursor = conn.cursor()
cursor.execute('''
    SELECT * FROM MEMBER WHERE NAME in ('홍길동','김길동')
''') # 홍길동, 김길동 자리에 입력값을 받아 쓰겠다 => 파라미터 사용하기
cursor.fetchall()

[('홍길동', 20, 'h@h.com'), ('김길동', 30, 'k@k.com')]

### qmark 방법으로 파라미터 사용하기

In [65]:
conn = sqlite3.connect('data/ch15_example.db')
cursor = conn.cursor()
sql = "SELECT * FROM MEMBER WHERE NAME in (?,?)"
name1 = input('검색할 이름은?')
name2 = input('검색할 다른 이름은?')
cursor.execute(sql, (name1, name2)) # 튜플로 기입
cursor.fetchall()

검색할 이름은?성춘향
검색할 다른 이름은?홍길동


[('홍길동', 20, 'h@h.com')]

### 추천) named 방법으로 파라미터 사용하기

In [67]:
sql = "SELECT * FROM MEMBER WHERE NAME in (:name1,:name2)"
name1 = input('검색할 이름은?')
name2 = input('검색할 다른 이름은?')
cursor.execute(sql, {'name1':name1, 'name2':name2}) # 딕셔너리로 기입
cursor.fetchall()

검색할 이름은?김길동
검색할 다른 이름은?김길동


[('김길동', 30, 'k@k.com')]

In [69]:
# named 방법을 이용해서 데이터 입력하기
try :
    name = input('입력할 이름은 : ')
    age = int(input('입력할 나이는?'))
except ValueError :
    print('유효하지 않은 나이를 입력하시면 18세로 초기화')
    age = 18
finally :
    email = input('사용할 email은?')
cursor.execute("INSERT INTO MEMBER VALUES (:NAME,:AGE,:EMAIL)",{'NAME':name,'AGE':age,'EMAIL':email})
conn.commit()
if cursor.rowcount == 1 :
    print('성공')
else :
    print('실패')

입력할 이름은 : 정종혁
입력할 나이는?30
사용할 email은?wjdwhdgur111@naver.com
성공


In [70]:
cursor.close()
conn.close()

# 2절. 오라클 데이터베이스 연결
- pip install cx_Oacle

In [72]:
import cx_Oracle
# conn 얻어오는 방법1
conn = cx_Oracle.connect('scott','tiger','localhost:1521/xe')
conn

<cx_Oracle.Connection to scott@localhost:1521/xe>

In [76]:
# conn 얻어오는 방법2
oracle_dsn = cx_Oracle.makedsn(host = 'localhost',port=1521, sid='xe')
conn = cx_Oracle.connect('scott','tiger',dsn=oracle_dsn)
conn

<cx_Oracle.Connection to scott@(DESCRIPTION=(ADDRESS=(PROTOCOL=TCP)(HOST=localhost)(PORT=1521))(CONNECT_DATA=(SID=xe)))>

In [77]:
cursor = conn.cursor()

In [80]:
sql = "SELECT EMPNO NO, ENAME, JOB, MGR, SAL, COMM FROM EMP"
cursor.execute(sql)
emps = cursor.fetchall()
for emp in emps :
    print(emp) # none == NULL

(7369, 'SMITH', 'CLERK', 7902, 800.0, None)
(7499, 'ALLEN', 'SALESMAN', 7698, 1600.0, 300.0)
(7521, 'WARD', 'SALESMAN', 7698, 1250.0, 500.0)
(7566, 'JONES', 'MANAGER', 7839, 2975.0, None)
(7654, 'MARTIN', 'SALESMAN', 7698, 1250.0, 1400.0)
(7698, 'BLAKE', 'MANAGER', 7839, 2850.0, None)
(7782, 'CLARK', 'MANAGER', 7839, 2450.0, None)
(7788, 'SCOTT', 'ANALYST', 7566, 3000.0, None)
(7839, 'KING', 'PRESIDENT', None, 5000.0, None)
(7844, 'TURNER', 'SALESMAN', 7698, 1500.0, 0.0)
(7876, 'ADAMS', 'CLERK', 7788, 1100.0, None)
(7900, 'JAMES', 'CLERK', 7698, 950.0, None)
(7902, 'FORD', 'ANALYST', 7566, 3000.0, None)
(7934, 'MILLER', 'CLERK', 7782, 1300.0, None)


In [81]:
pd.DataFrame(emps, columns = [des[0] for des in cursor.description])

,NO,ENAME,JOB,MGR,SAL,COMM
0,7369,SMITH,CLERK,7902.0,800.0,NaN
1,7499,ALLEN,SALESMAN,7698.0,1600.0,300.0
2,7521,WARD,SALESMAN,7698.0,1250.0,500.0
3,7566,JONES,MANAGER,7839.0,2975.0,NaN
4,7654,MARTIN,SALESMAN,7698.0,1250.0,1400.0
5,7698,BLAKE,MANAGER,7839.0,2850.0,NaN
6,7782,CLARK,MANAGER,7839.0,2450.0,NaN
7,7788,SCOTT,ANALYST,7566.0,3000.0,NaN
8,7839,KING,PRESIDENT,NaN,5000.0,NaN
9,7844,TURNER,SALESMAN,7698.0,1500.0,0.0


# 3절. MySql 연결하기

In [1]:
%pip install pymysql

     ---------------------------------------- 45.0/45.0 kB 2.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [14]:
import pymysql

conn = pymysql.connect(
    host="localhost",
    user="root",
    password="mysql",
    database="devdb"
)
data = []
with conn.cursor() as cursor:
    cursor.execute("SELECT * FROM personal WHERE startdate = '1990-12-17';")
    for row in cursor.fetchall():
        data.append(row)
    cursor.execute("DESCRIBE personal")
    columns = [row[0] for row in cursor.fetchall()]

conn.close()

In [15]:
import pandas as pd
data = pd.DataFrame(data)
display(data)
columns

,0,1,2,3,4,5,6,7
0,1111,smith,manager,1001,1990-12-17,1000,None,10


['pno', 'pname', 'job', 'manager', 'startdate', 'pay', 'bonus', 'dno']